# 01. Exploratory Data Analysis (EDA) & Text Preprocessing
**Proyek:** Analisis Klasifikasi Ujaran Kebencian pada Informasi Pemerintahan Indonesia di Media Sosial Menggunakan Machine Learning dan Deep Learning  
**Keterkaitan SDG:** **SDG 16** (*Peace, Justice and Strong Institutions*) — Target 16.10 & 16.7  
**Peneliti:** Ardhana Syah Hidayat & Muhammad Adam Al-Kidri  

---

### Tujuan Notebook:
1. **Pemeriksaan Data & Integritas:** Mengidentifikasi distribusi data mentah dan menghapus duplikasi (*exact duplicates*) untuk mencegah *Data Leakage*.
2. **Ekstraksi Fitur Gaya Penulisan (*Pre-Cleaning Feature Engineering*):** Mengekstrak indikator perilaku teks seperti rasio huruf kapital (*shouting*), jumlah tanda baca, panjang kalimat, dan kepadatan kata kasar sebelum dilakukan *lowercasing*.
3. **Pembersihan Teks & Normalisasi Kamus Alay:** Mengubah kata-kata *slang/alay* menjadi bahasa Indonesia baku serta menghapus token anonim Twitter (`USER`, `URL`, `RT`).
4. **Pemisahan Teks Kontekstual vs Leksikal:** Menghasilkan dua versi teks bersih:
   * `text_clean_contextual` (untuk IndoBERT / Deep Learning — mempertahankan kata negasi dan struktur kalimat).
   * `text_clean_lexical` (untuk Logistic Regression & XGBoost — stopword filtering dengan pengecualian kata negasi penting).
5. **Analisis Eksplorasi Data (EDA):** Menganalisis perbedaan karakteristik linguistik antara ujaran kebencian (*Hate Speech*) dan opini/kritik wajar, khususnya pada diskursus pemerintahan.
6. **Pembagian Data Terstratifikasi (*Stratified Split*):** Membagi dataset menjadi 70% Train, 15% Validation, dan 15% Test Set secara konsisten (`random_state=42`).

In [ ]:
import os
import re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

# Konfigurasi visualisasi
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 120

print("Library berhasil dimuat.")

## 1. Pemuatan Dataset Mentah & Kamus Pendukung
Dataset yang digunakan adalah *Indonesian Abusive and Hate Speech Twitter Text Dataset* (Ibrohim & Budi, 2019) yang dilengkapi dengan kamus leksikon kata kasar (`abusive.csv`) dan kamus normalisasi kata tidak baku (`new_kamusalay.csv`).

In [ ]:
# Penentuan path adaptif (baik dijalankan dari root maupun folder notebooks)
current_dir = Path.cwd()
root_dir = current_dir if (current_dir / 'data').exists() else current_dir.parent
data_dir = root_dir / 'data'
processed_dir = data_dir / 'processed'
processed_dir.mkdir(parents=True, exist_ok=True)

df_raw = pd.read_csv(data_dir / 'data.csv', encoding='latin1')
kamus_df = pd.read_csv(data_dir / 'new_kamusalay.csv', encoding='latin1', header=None, names=['slang', 'formal'])
abusive_df = pd.read_csv(data_dir / 'abusive.csv', encoding='latin1')

print(f"Dimensi data mentah: {df_raw.shape}")
print(f"Jumlah entri kamus alay: {len(kamus_df)}")
print(f"Jumlah leksikon kata kasar: {len(abusive_df)}")
df_raw.head(3)

## 2. Deduplikasi Teks (*Exact Duplicate Removal*)
Mencegah *Data Leakage*: jika terdapat teks yang 100% identik di beberapa baris cuitan (misalnya hasil *retweet* atau spam), teks tersebut berisiko masuk ke dalam *Train Set* dan *Test Set* secara bersamaan. Oleh karena itu, kita lakukan deduplikasi berbasis kolom `Tweet`.

In [ ]:
initial_rows = len(df_raw)
df = df_raw.drop_duplicates(subset=['Tweet']).reset_index(drop=True)
duplicate_count = initial_rows - len(df)

print(f"Baris sebelum deduplikasi : {initial_rows}")
print(f"Baris setelah deduplikasi : {len(df)}")
print(f"Jumlah duplikat dihapus   : {duplicate_count} ({duplicate_count/initial_rows:.2%})")

## 3. Ekstraksi Fitur Gaya Penulisan (*Pre-Cleaning Feature Engineering*)
> [!IMPORTANT]  
> Tahap ini **WAJIB** dieksekusi sebelum teks diubah ke huruf kecil (*lowercase*) atau dibersihkan dari tanda baca, agar informasi penulisan asli (seperti teriakan huruf kapital dan intensitas tanda baca) tidak hilang.

In [ ]:
# 1. char_length: Panjang karakter teks asli
df['char_length'] = df['Tweet'].astype(str).str.len()

# 2. word_count: Jumlah token/kata asli
df['word_count'] = df['Tweet'].astype(str).str.split().str.len()

# 3. caps_ratio: Rasio huruf kapital terhadap panjang karakter
df['caps_ratio'] = df['Tweet'].apply(
    lambda x: sum(1 for c in str(x) if c.isupper()) / (len(str(x)) + 1)
)

# 4. exclamation_count: Jumlah kemunculan tanda seru (!) dan tanda tanya (?)
df['exclamation_count'] = df['Tweet'].apply(
    lambda x: sum(1 for c in str(x) if c in ['!', '?'])
)

# 5. user_mention_count: Frekuensi mention akun lain (token USER)
df['user_mention_count'] = df['Tweet'].apply(
    lambda x: len(re.findall(r'\bUSER\b', str(x)))
)

df[['Tweet', 'char_length', 'word_count', 'caps_ratio', 'exclamation_count', 'user_mention_count']].head(3)

## 4. Pembersihan Teks & Normalisasi Kamus Alay
Sesuai spesifikasi arsitektur tiga model di `CONTEXT.md`:
1. `text_clean_contextual`: Untuk **Model 3 (IndoBERT)** — dibersihkan dari token anonim Twitter, format hex byte, dan tanda baca, dinormalisasi dengan kamusalay, namun **tetap mempertahankan stopwords** agar makna negasi dan sintaksis kalimat utuh.
2. `text_clean_lexical`: Untuk **Model 1 (Logistic Regression)** dan **Model 2 (XGBoost)** — teks bersih dengan penghapusan stopwords umum, **kecuali kata negasi krusial** (`tidak`, `bukan`, `jangan`, `belum`, `tak`, dll.) agar sentimen tidak terbalik.

In [ ]:
# Hash Map pencarian cepat kata alay & kata kasar
slang_dict = dict(zip(kamus_df['slang'].str.strip(), kamus_df['formal'].str.strip()))
abusive_set = set(abusive_df['ABUSIVE'].str.lower().str.strip().tolist())

def clean_and_normalize(text):
    text = str(text).lower()
    # Hapus token anonimisasi Twitter
    text = re.sub(r'\buser\b|\burl\b|\brt\b', ' ', text)
    # Hapus kode byte emoji mentah (\xf0...)
    text = re.sub(r'\\x[0-9a-f]{2}', ' ', text)
    # Hapus karakter selain huruf, angka, dan spasi
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    # Normalisasi slang menggunakan kamusalay
    words = [slang_dict.get(w, w) for w in text.split()]
    return ' '.join(words)

# Buat versi kontekstual
df['text_clean_contextual'] = df['Tweet'].apply(clean_and_normalize)

# Hitung jumlah leksikon kata kasar berdasarkan teks yang telah dinormalisasi
df['abusive_lexicon_count'] = df['text_clean_contextual'].apply(
    lambda x: sum(1 for w in x.split() if w in abusive_set)
)

# Buat versi leksikal (penyaringan stopword dengan proteksi kata negasi)
sastrawi_factory = StopWordRemoverFactory()
default_stopwords = set(sastrawi_factory.get_stop_words())
negation_words = {'tidak', 'bukan', 'jangan', 'belum', 'tak', 'tiada', 'tanpa'}
lexical_stopwords = default_stopwords - negation_words

def remove_stopwords_with_negation(text):
    words = [w for w in text.split() if w not in lexical_stopwords]
    return ' '.join(words)

df['text_clean_lexical'] = df['text_clean_contextual'].apply(remove_stopwords_with_negation)

# Flag topik pemerintahan untuk keperluan audit SDG 16 (Bukan fitur X!)
gov_keywords = [
    'presiden', 'jokowi', 'prabowo', 'ahok', 'anies', 'pemerintah', 'menteri',
    'dpr', 'negara', 'rakyat', 'rezim', 'partai', 'politik', 'kpk', 'polri',
    'polisi', 'korupsi', 'pajak', 'undang', 'perppu', 'gubernur', 'pilkada', 'nkri'
]
gov_regex = r'\b(?:' + '|'.join(gov_keywords) + r')\b'
df['is_gov'] = df['text_clean_contextual'].str.contains(gov_regex, regex=True).astype(int)

print("Contoh Perbandingan Teks:")
print(f"Mentah      : {df['Tweet'].iloc[1]}")
print(f"Kontekstual : {df['text_clean_contextual'].iloc[1]}")
print(f"Leksikal    : {df['text_clean_lexical'].iloc[1]}")

## 5. Eksplorasi Data & Visualisasi (EDA)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Bar plot
sns.countplot(x='HS', data=df, ax=axes[0], palette=['#4CAF50', '#F44336'])
axes[0].set_title('Distribusi Kelas Target (HS)', fontsize=13, fontweight='bold')
axes[0].set_xticklabels(['Non-Hate Speech (0)', 'Hate Speech (1)'])
axes[0].set_ylabel('Jumlah Sampel')
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height()):,}\n({p.get_height()/len(df):.1%})',
                     (p.get_x() + p.get_width() / 2., p.get_height() / 2),
                     ha='center', va='center', color='white', fontweight='bold')

# Pie chart
hs_counts = df['HS'].value_counts()
axes[1].pie(hs_counts, labels=['Non-Hate Speech', 'Hate Speech'],
            autopct='%1.1f%%', colors=['#4CAF50', '#F44336'], startangle=140,
            explode=(0.04, 0), textprops={'fontsize': 11, 'fontweight': 'bold'})
axes[1].set_title('Proporsi Kelas Target', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

### 5.2 Analisis Gaya Penulisan terhadap Ujaran Kebencian
Apakah teks ujaran kebencian cenderung memiliki rasio huruf kapital lebih tinggi, tanda seru/tanya lebih banyak, atau kata kasar lebih padat?

In [ ]:
features_to_plot = [
    ('caps_ratio', 'Rasio Huruf Kapital (Caps Ratio)'),
    ('exclamation_count', 'Jumlah Tanda Seru & Tanya (! / ?)'),
    ('abusive_lexicon_count', 'Frekuensi Kata Kasar (Abusive Lexicon Count)'),
    ('char_length', 'Panjang Karakter (Character Length)')
]

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.flatten()

for idx, (col, title) in enumerate(features_to_plot):
    sns.boxplot(x='HS', y=col, data=df, ax=axes[idx], palette=['#81C784', '#E57373'], showfliers=False)
    axes[idx].set_title(title, fontsize=12, fontweight='bold')
    axes[idx].set_xticklabels(['Non-Hate Speech (0)', 'Hate Speech (1)'])
    axes[idx].set_ylabel('Nilai')

plt.tight_layout()
plt.show()

### 5.3 Analisis Diskursus Pemerintahan & Kebijakan Publik (SDG 16)
Pemeriksaan proporsi ujaran kebencian vs opini/kritik wajar pada cuitan yang memuat kata kunci pemerintahan/politik (`is_gov`).

In [ ]:
gov_ct = pd.crosstab(df['is_gov'], df['HS'], normalize='index') * 100
print("Tabel Silang Proporsi (%) Topik Pemerintahan vs Target HS:")
print(gov_ct.rename(index={0: 'Non-Pemerintahan', 1: 'Topik Pemerintahan'},
                    columns={0: 'Non-HS (%)', 1: 'Hate Speech (%)'}))

fig, ax = plt.subplots(figsize=(8, 4.5))
gov_ct.plot(kind='bar', stacked=True, color=['#4CAF50', '#F44336'], ax=ax)
ax.set_title('Proporsi Ujaran Kebencian pada Topik Pemerintahan vs Non-Pemerintahan', fontsize=12, fontweight='bold')
ax.set_xticklabels(['Cuitan Non-Pemerintahan', 'Cuitan Bertema Pemerintahan'], rotation=0)
ax.set_ylabel('Persentase (%)')
ax.legend(['Non-Hate Speech (0)', 'Hate Speech (1)'], loc='upper right')

for p in ax.patches:
    width, height = p.get_width(), p.get_height()
    if height > 5:
        ax.annotate(f'{height:.1f}%', (p.get_x() + width / 2, p.get_y() + height / 2),
                    ha='center', va='center', color='white', fontweight='bold')

plt.tight_layout()
plt.show()

### 5.4 Analisis N-Gram Kata Teratas
Melihat kata-kata yang paling dominan muncul pada masing-masing kelas setelah pembersihan teks.

In [ ]:
def get_top_ngrams(corpus, n=1, top_k=15):
    vec = CountVectorizer(ngram_range=(n, n), max_features=top_k).fit(corpus)
    bag_of_words = vec.transform(corpus)
    sum_words = bag_of_words.sum(axis=0)
    words_freq = [(word, sum_words[0, idx]) for word, idx in vec.vocabulary_.items()]
    return sorted(words_freq, key=lambda x: x[1], reverse=True)

top_hs = get_top_ngrams(df[df['HS'] == 1]['text_clean_lexical'], n=1, top_k=12)
top_non_hs = get_top_ngrams(df[df['HS'] == 0]['text_clean_lexical'], n=1, top_k=12)

df_top_hs = pd.DataFrame(top_hs, columns=['Kata', 'Frekuensi'])
df_top_non_hs = pd.DataFrame(top_non_hs, columns=['Kata', 'Frekuensi'])

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.barplot(x='Frekuensi', y='Kata', data=df_top_hs, ax=axes[0], palette='Reds_r')
axes[0].set_title('Kata Teratas pada Ujaran Kebencian (HS = 1)', fontsize=12, fontweight='bold')

sns.barplot(x='Frekuensi', y='Kata', data=df_top_non_hs, ax=axes[1], palette='Greens_r')
axes[1].set_title('Kata Teratas pada Non-Ujaran Kebencian (HS = 0)', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

## 6. Pembagian Data Terstratifikasi (*Stratified Split: 70:15:15*)
Sesuai metodologi di `CONTEXT.md`:
* **70% Training Set:** Untuk pelatihan representasi TF-IDF, reduksi SVD, dan parameter model.
* **15% Validation Set:** Untuk penyetelan hiperparameter regularisasi dan *early stopping*.
* **15% Testing Set:** Untuk evaluasi akhir yang tidak tersentuh saat pelatihan.

In [ ]:
columns_order = [
    'Tweet', 'text_clean_contextual', 'text_clean_lexical',
    'char_length', 'word_count', 'caps_ratio', 'exclamation_count',
    'user_mention_count', 'abusive_lexicon_count', 'is_gov',
    'HS', 'Abusive', 'HS_Individual', 'HS_Group', 'HS_Religion',
    'HS_Race', 'HS_Physical', 'HS_Gender', 'HS_Other',
    'HS_Weak', 'HS_Moderate', 'HS_Strong'
]
df_final = df[columns_order]

train_df, temp_df = train_test_split(
    df_final, test_size=0.30, stratify=df_final['HS'], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df['HS'], random_state=42
)

# Simpan ke data/processed/
train_path = processed_dir / 'train.csv'
val_path = processed_dir / 'val.csv'
test_path = processed_dir / 'test.csv'

train_df.to_csv(train_path, index=False)
val_df.to_csv(val_path, index=False)
test_df.to_csv(test_path, index=False)

print("Ringkasan Pembagian Data Terstratifikasi:")
print(f"• Training Set   : {len(train_df):,} baris ({len(train_df)/len(df_final):.1%}) | Proporsi HS: {train_df['HS'].mean():.2%}")
print(f"• Validation Set : {len(val_df):,} baris ({len(val_df)/len(df_final):.1%}) | Proporsi HS: {val_df['HS'].mean():.2%}")
print(f"• Testing Set    : {len(test_df):,} baris ({len(test_df)/len(df_final):.1%}) | Proporsi HS: {test_df['HS'].mean():.2%}")
print(f"\nData siap digunakan untuk pelatihan ketiga model di tahap selanjutnya.")